# Biotech Gender Pay Parity Analysis
## CMS Open Payments 2014 to 2024

This notebook analyzes gender disparities in biotech industry payments to U.S. physicians
using the CMS Open Payments database across ten program years.

**Runtime:** Google Colab (free tier)

| Section | What it does |
| --- | --- |
| 1 to 4 | Setup, download 10 years of CMS data + NPPES, ETL to Parquet, export CSV |
| 5 to 6 | Load data, apply specialty mapping, compute KPIs |
| 7 | Gender distribution: 3 charts (map, specialty, seniority) |
| 8 | Payment distribution: 3 charts (R/Y/G map, specialty+gender, seniority+gender) |
| 9 | Statistical tests (Mann-Whitney, Welch, Cohen's d) |
| 10 | Control models: 5 linear + 5 logistic (prove gender matters) |
| 11 | All-variable models: 5 linear + 5 logistic (maximize prediction) |
| 12 | Model evaluation dashboard (comparisons + coefficient gauge) |
| 13 | Streamlit app generation + deployment guide |


---
## 1. Setup


In [ ]:
!pip install -q duckdb pyarrow xgboost


Import everything needed for the analysis. DuckDB handles multi-GB files without
loading them into memory, which is essential when processing a decade of CMS data on Colab.


In [ ]:
import duckdb, os, time, subprocess, zipfile, warnings, glob, gc, shutil
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.linear_model import (
    LogisticRegression, LinearRegression, Ridge, Lasso, ElasticNet
)
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score,
    mean_squared_error, r2_score, f1_score, confusion_matrix
)
from scipy import stats
import xgboost as xgb
warnings.filterwarnings('ignore')

# Colab paths
RAW = '/content/raw'
PQ = '/content/pq'
NPPES_DIR = '/content/nppes'
for d in [RAW, PQ, NPPES_DIR]:
    os.makedirs(d, exist_ok=True)

con = duckdb.connect('/content/op.duckdb')
con.execute("SET memory_limit='10GB'; SET threads=2")

# CPI-U annual averages for inflation adjustment (all amounts converted to 2024 dollars)
CPI = {
    2014: 236.736, 2015: 237.017, 2016: 240.007, 2017: 245.120,
    2018: 251.107, 2019: 255.657, 2020: 258.811, 2021: 270.970,
    2022: 292.655, 2023: 304.702, 2024: 314.690
}
INFLATION = {yr: 314.690 / cpi for yr, cpi in CPI.items()}

# DOOIT study protocol: four specialty categories
STUDY_SPECS = {
    'Surgery': ['surgery', 'surgical', 'orthopedic', 'orthopaedic'],
    'Oncology': ['oncology', 'hematology/oncology', 'medical oncology', 'surgical oncology'],
    'Cardiology': ['cardiology', 'cardiovascular', 'interventional cardiology'],
    'Neurology': ['neurology', 'neurological', 'neurosurgery']
}

def assign_specialty(raw_spec):
    if not raw_spec:
        return None
    lower = raw_spec.lower()
    for category, keywords in STUDY_SPECS.items():
        if any(kw in lower for kw in keywords):
            return category
    return None

def download_file(url, dest):
    print(f'  Downloading {os.path.basename(dest)}...', end=' ')
    subprocess.run(
        ['wget', '-q', '--tries=3', '--timeout=300', '-O', dest, url],
        capture_output=True
    )
    if os.path.exists(dest) and os.path.getsize(dest) > 10000:
        print(f'{os.path.getsize(dest) / 1e9:.2f} GB')
        return True
    if os.path.exists(dest):
        os.remove(dest)
    print('FAILED')
    return False

print('Setup complete.')
for yr in sorted(INFLATION):
    print(f'  {yr}: multiply by {INFLATION[yr]:.4f} to convert to 2024 dollars')


---
## 2. Download NPPES Registry

The NPPES file contains each physician's self-reported gender at the time of NPI registration.
We extract only the NPI and sex code into a compact Parquet file, then delete the source.
All gender assignment in this analysis comes from this single field via INNER JOIN.
No name-based inference is used at any point.


In [ ]:
NPPES_URL = 'https://download.cms.gov/nppes/NPPES_Data_Dissemination_March_2026_V2.zip'
NPPES_ZIP = os.path.join(NPPES_DIR, 'nppes.zip')
NPPES_PQ = os.path.join(NPPES_DIR, 'nppes_sex.parquet')

if os.path.exists(NPPES_PQ):
    count = con.execute(f"SELECT COUNT(*) FROM '{NPPES_PQ}'").fetchone()[0]
    print(f'NPPES already processed: {count:,} providers')
else:
    if not os.path.exists(NPPES_ZIP):
        download_file(NPPES_URL, NPPES_ZIP)

    if os.path.exists(NPPES_ZIP):
        with zipfile.ZipFile(NPPES_ZIP, 'r') as zf:
            csv_names = [f for f in zf.namelist() if f.endswith('.csv') and 'npidata' in f.lower()]
            if not csv_names:
                csv_names = sorted(
                    [f for f in zf.namelist() if f.endswith('.csv')],
                    key=lambda f: zf.getinfo(f).file_size, reverse=True
                )
            zf.extract(csv_names[0], NPPES_DIR)
            csv_path = os.path.join(NPPES_DIR, csv_names[0])

        con.execute(f"""
            COPY (
                SELECT
                    CAST(NPI AS VARCHAR) AS npi,
                    \"Provider Sex Code\" AS sex
                FROM read_csv_auto('{csv_path}',
                    header=true, sample_size=50000,
                    ignore_errors=true, all_varchar=true)
                WHERE \"Entity Type Code\" = '1'
                  AND \"Provider Sex Code\" IN ('M', 'F')
            ) TO '{NPPES_PQ}' (FORMAT PARQUET, COMPRESSION ZSTD)
        """)

        count = con.execute(f"SELECT COUNT(*) FROM '{NPPES_PQ}'").fetchone()[0]
        print(f'Processed: {count:,} providers')

        # Free disk
        os.remove(csv_path)
        if os.path.exists(NPPES_ZIP):
            os.remove(NPPES_ZIP)

sex_dist = con.execute(f"SELECT sex, COUNT(*) n FROM '{NPPES_PQ}' GROUP BY sex ORDER BY n DESC").fetchdf()
print(sex_dist.to_string(index=False))


---
## 3. Download and Process CMS Open Payments (2014 to 2024)

For each program year, we download the ZIP from CMS, extract the General Payments CSV,
run the ETL pipeline through DuckDB (biotech filter, NPPES gender join, inflation adjustment),
write to compressed Parquet, and then immediately delete the raw CSV and ZIP to free disk space.
This is necessary because the combined raw data exceeds 50 GB.


In [ ]:
# Biotech identification: keyword matching on the company name field
BIOTECH_KEYWORDS = [
    'biotech', 'biotherapeutics', 'biosciences', 'biopharmaceuticals',
    'biologics', 'biopharma', 'bioscience', 'genomics',
    'gene therapy', 'therapeutics', 'oncology'
]

COMPANY_COL = 'Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name'
keyword_sql = ' OR '.join([
    f"UPPER(\"{COMPANY_COL}\") LIKE '%{kw.upper()}%'"
    for kw in BIOTECH_KEYWORDS
])
biotech_filter = f'({keyword_sql})'

COMBINED_PQ = os.path.join(PQ, 'bio_2014_2024.parquet')

if os.path.exists(COMBINED_PQ):
    n = con.execute(f"SELECT COUNT(*) FROM '{COMBINED_PQ}'").fetchone()[0]
    print(f'Combined parquet already exists: {n:,} rows')
else:
    year_parquets = []
    for year in range(2014, 2025):
        url = f'https://download.cms.gov/openpayments/PGYR{year}_P01232026_01102026.zip'
        zip_path = os.path.join(RAW, f'op_{year}.zip')
        year_pq = os.path.join(PQ, f'bio_{year}.parquet')
        multiplier = INFLATION.get(year, 1.0)

        print(f'\n--- {year} (inflation x{multiplier:.4f}) ---')

        # Find or download CSV
        csv_files = glob.glob(os.path.join(RAW, f'*{year}*.csv'))
        if not csv_files:
            if not os.path.exists(zip_path):
                if not download_file(url, zip_path):
                    continue
            try:
                with zipfile.ZipFile(zip_path, 'r') as zf:
                    gnrl = [f for f in zf.namelist() if 'GNRL' in f.upper() and f.endswith('.csv')]
                    target = gnrl[0] if gnrl else max(
                        [f for f in zf.namelist() if f.endswith('.csv')],
                        key=lambda f: zf.getinfo(f).file_size
                    )
                    zf.extract(target, RAW)
                    print(f'  Extracted: {target}')
            except Exception as e:
                print(f'  Extract error: {e}')
                continue
            finally:
                if os.path.exists(zip_path):
                    os.remove(zip_path)

            csv_files = glob.glob(os.path.join(RAW, f'*{year}*.csv'))
            if not csv_files:
                csv_files = [f for f in glob.glob(os.path.join(RAW, '*.csv'))
                             if str(year) in os.path.basename(f)]

        if not csv_files:
            print(f'  No CSV found, skipping')
            continue

        csv_path = csv_files[0]

        # ETL: biotech filter + gender join + inflation
        try:
            t0 = time.time()
            con.execute(f"""
                COPY (
                    SELECT
                        CAST(op.Covered_Recipient_NPI AS VARCHAR) AS npi,
                        op.Covered_Recipient_First_Name AS first_name,
                        op.Covered_Recipient_Last_Name AS last_name,
                        op.Covered_Recipient_Specialty_1 AS specialty_raw,
                        op.Recipient_State AS state,
                        op.\"{COMPANY_COL}\" AS company,
                        ROUND(CAST(op.Total_Amount_of_Payment_USDollars AS DOUBLE) * {multiplier}, 2) AS amt,
                        CAST(op.Total_Amount_of_Payment_USDollars AS DOUBLE) AS amt_nominal,
                        op.Nature_of_Payment_or_Transfer_of_Value AS pay_type,
                        nppes.sex AS gender,
                        {year} AS program_year
                    FROM read_csv_auto('{csv_path}',
                        header=true, sample_size=100000,
                        ignore_errors=true, all_varchar=true) op
                    INNER JOIN '{NPPES_PQ}' nppes
                        ON CAST(op.Covered_Recipient_NPI AS VARCHAR) = nppes.npi
                    WHERE op.Covered_Recipient_Type = 'Covered Recipient Physician'
                      AND {biotech_filter}
                ) TO '{year_pq}' (FORMAT PARQUET, COMPRESSION ZSTD)
            """)
            n = con.execute(f"SELECT COUNT(*) FROM '{year_pq}'").fetchone()[0]
            print(f'  {n:,} rows in {time.time() - t0:.0f}s')
            year_parquets.append(year_pq)
        except Exception as e:
            print(f'  ETL error: {e}')

        # Delete raw CSV immediately
        os.remove(csv_path)
        gc.collect()

    # Combine all years into one file
    if year_parquets:
        union_query = ' UNION ALL '.join([f"SELECT * FROM '{p}'" for p in year_parquets])
        con.execute(f"COPY ({union_query}) TO '{COMBINED_PQ}' (FORMAT PARQUET, COMPRESSION ZSTD)")
        for p in year_parquets:
            os.remove(p)
        n = con.execute(f"SELECT COUNT(*) FROM '{COMBINED_PQ}'").fetchone()[0]
        print(f'\nCombined: {n:,} rows, {os.path.getsize(COMBINED_PQ) / 1e6:.1f} MB')


---
## 4. Export Clean CSV and Free Disk Space

Write a single analysis-ready CSV with all relevant columns, then delete everything
except the CSV and Parquet files.


In [ ]:
con.execute(f"CREATE OR REPLACE VIEW raw_all AS SELECT * FROM '{COMBINED_PQ}'")

CSV_PATH = '/content/biotech_payments_2014_2024.csv'
con.execute(f"""
    COPY (
        SELECT * FROM raw_all ORDER BY program_year, gender, state
    ) TO '{CSV_PATH}' (HEADER, DELIMITER ',')
""")

total_rows = con.execute('SELECT COUNT(*) FROM raw_all').fetchone()[0]
print(f'Exported: {total_rows:,} rows, {os.path.getsize(CSV_PATH) / 1e6:.1f} MB')

# Clean up raw directory
if os.path.exists(RAW):
    shutil.rmtree(RAW)
    print('Raw files deleted.')


---
## 5. Load Data and Prepare for Analysis

Apply the DOOIT study specialty mapping, build per-physician aggregates,
and identify top companies dynamically from the data.


In [ ]:
all_data = con.execute('SELECT * FROM raw_all').fetchdf()
all_data['spec_cat'] = all_data['specialty_raw'].apply(assign_specialty)
all_data['gender_label'] = all_data['gender'].map({'M': 'Male', 'F': 'Female'})

# Keep only the four study specialties
df = all_data[all_data['spec_cat'].notna()].copy()

# Per-physician summary
per_doc = df.groupby(
    ['npi', 'gender', 'gender_label', 'spec_cat', 'state', 'first_name', 'last_name']
).agg(
    n_pay=('amt', 'count'),
    total=('amt', 'sum'),
    avg_pay=('amt', 'mean'),
    max_pay=('amt', 'max'),
    n_companies=('company', 'nunique'),
    n_years=('program_year', 'nunique'),
    first_year=('program_year', 'min'),
    last_year=('program_year', 'max')
).reset_index()

per_doc['career_span'] = per_doc['last_year'] - per_doc['first_year'] + 1
per_doc['seniority'] = pd.cut(
    per_doc['career_span'],
    bins=[0, 1, 3, 5, 8, 11],
    labels=['1 yr', '2-3 yr', '4-5 yr', '6-8 yr', '9-11 yr'],
    right=True, include_lowest=True
)

# Top companies from the data (not hardcoded)
top_20_companies = df['company'].value_counts().head(20).index.tolist()
top_5_companies = df['company'].value_counts().head(5).index
top_15_states = df['state'].value_counts().head(15).index
top_10_pay_types = df['pay_type'].value_counts().head(10).index

print(f'All biotech records: {len(all_data):,}')
print(f'Study specialties:   {len(df):,}')
print(f'Physicians:          {len(per_doc):,}')
print(f'Years covered:       {sorted(df.program_year.unique())}')
print(f'\nSpecialty breakdown:')
print(df['spec_cat'].value_counts().to_string())


---
## 6. KPI Summary

Four headline numbers with female-share breakdowns.


In [ ]:
total_records = len(df)
female_records = len(df[df['gender'] == 'F'])
pct_female_records = round(100 * female_records / total_records, 1)

total_dollars = df['amt'].sum()
female_dollars = df[df['gender'] == 'F']['amt'].sum()
pct_female_dollars = round(100 * female_dollars / total_dollars, 1)

male_median_overall = all_data[all_data['gender'] == 'M']['amt'].median()
female_median_overall = all_data[all_data['gender'] == 'F']['amt'].median()
fm_ratio_overall = round(female_median_overall / male_median_overall, 3)

male_median_study = df[df['gender'] == 'M']['amt'].median()
female_median_study = df[df['gender'] == 'F']['amt'].median()
fm_ratio_study = round(female_median_study / male_median_study, 3)

print(f'KPI 1 - Records:           {total_records:,} ({pct_female_records}% female)')
print(f'KPI 2 - Total amount:      ${total_dollars:,.0f} ({pct_female_dollars}% female)')
print(f'KPI 3 - F/M median overall: {fm_ratio_overall}')
print(f'KPI 4 - F/M median biotech: {fm_ratio_study}')

fig = make_subplots(
    rows=1, cols=4,
    specs=[[{'type': 'indicator'}] * 4],
    subplot_titles=[
        f'Records ({pct_female_records}% F)',
        f'Total $ ({pct_female_dollars}% F)',
        'F/M Median Overall',
        'F/M Median Biotech'
    ]
)
for i, (val, fmt) in enumerate([
    (total_records, ',.0f'),
    (total_dollars, '$,.0f'),
    (fm_ratio_overall, '.3f'),
    (fm_ratio_study, '.3f')
]):
    fig.add_trace(
        go.Indicator(mode='number', value=val, number={'valueformat': fmt, 'font': {'size': 36}}),
        row=1, col=i + 1
    )
fig.update_layout(height=150, margin=dict(t=40, b=10))
fig.show()


---
## 7. Gender Distribution

Three views of how physicians are distributed by gender across geography,
specialty, and career seniority.


### 7.1 Across Regions


In [ ]:
state_gender = df.groupby(['state', 'gender_label']).agg(
    physicians=('npi', 'nunique')
).reset_index()

state_pivot = state_gender.pivot_table(
    index='state', columns='gender_label', values='physicians', fill_value=0
).reset_index()

if 'Female' in state_pivot.columns and 'Male' in state_pivot.columns:
    state_pivot['pct_female'] = (
        100 * state_pivot['Female'] / (state_pivot['Female'] + state_pivot['Male'])
    ).round(1)
    state_pivot['total'] = state_pivot['Female'] + state_pivot['Male']
    state_pivot = state_pivot[state_pivot['state'].str.len() == 2]

    fig = px.choropleth(
        state_pivot,
        locations='state', locationmode='USA-states',
        color='pct_female', scope='usa',
        color_continuous_scale='RdBu',
        range_color=[25, 50],
        title='Percentage of Female Physicians by State (2014 to 2024)',
        labels={'pct_female': '% Female'},
        hover_data={'total': ':,'}
    )
    fig.update_layout(height=500)
    fig.show()


### 7.2 Across Specialties


In [ ]:
spec_gender = df.groupby(['spec_cat', 'gender_label']).agg(
    physicians=('npi', 'nunique')
).reset_index()

fig = px.bar(
    spec_gender, x='spec_cat', y='physicians',
    color='gender_label', barmode='group',
    title='Physician Count by Specialty and Gender',
    labels={'physicians': 'Physicians', 'spec_cat': 'Specialty', 'gender_label': 'Gender'}
)
fig.show()


### 7.3 Across Seniority

Career seniority is proxied by the span of years a physician appears in CMS data.


In [ ]:
seniority_gender = per_doc.groupby(
    ['seniority', 'gender_label']
).size().reset_index(name='count')

fig = px.bar(
    seniority_gender, x='seniority', y='count',
    color='gender_label', barmode='group',
    title='Physician Count by Career Seniority and Gender',
    labels={'count': 'Physicians', 'seniority': 'Years in CMS Data', 'gender_label': 'Gender'},
    category_orders={'seniority': ['1 yr', '2-3 yr', '4-5 yr', '6-8 yr', '9-11 yr']}
)
fig.show()


---
## 8. Payment Distribution


### 8.1 Across Gender and Region (R/Y/G)

States colored by F/M mean payment ratio: green means women paid more (above 1.10),
gold means near parity (0.90 to 1.10), red means women paid less (below 0.90).


In [ ]:
state_pay = df.groupby(['state', 'gender']).agg(
    mean_pay=('amt', 'mean')
).reset_index()

state_pay_pivot = state_pay.pivot_table(
    index='state', columns='gender', values='mean_pay'
).reset_index()

state_totals = df.groupby('state')['amt'].sum().reset_index()
state_totals.columns = ['state', 'total_payments']

if 'F' in state_pay_pivot.columns and 'M' in state_pay_pivot.columns:
    state_pay_pivot['fm_ratio'] = (state_pay_pivot['F'] / state_pay_pivot['M']).round(3)
    state_pay_pivot = state_pay_pivot.merge(state_totals, on='state')
    state_pay_pivot = state_pay_pivot[state_pay_pivot['state'].str.len() == 2].dropna(subset=['fm_ratio'])

    state_pay_pivot['gap_category'] = state_pay_pivot['fm_ratio'].apply(
        lambda r: 'Women paid more (>1.10)' if r > 1.1
        else 'Near parity (0.90 to 1.10)' if r >= 0.9
        else 'Women paid less (<0.90)'
    )
    state_pay_pivot['total_label'] = state_pay_pivot['total_payments'].apply(
        lambda v: f'${v / 1e6:.1f}M' if v >= 1e6 else f'${v / 1e3:.0f}K'
    )

    fig = px.choropleth(
        state_pay_pivot,
        locations='state', locationmode='USA-states',
        color='gap_category', scope='usa',
        color_discrete_map={
            'Women paid less (<0.90)': 'crimson',
            'Near parity (0.90 to 1.10)': 'gold',
            'Women paid more (>1.10)': 'seagreen'
        },
        category_orders={
            'gap_category': ['Women paid less (<0.90)', 'Near parity (0.90 to 1.10)', 'Women paid more (>1.10)']
        },
        title='F/M Mean Payment Ratio by State',
        hover_data={'fm_ratio': ':.3f', 'total_label': True}
    )
    fig.update_layout(height=500)
    fig.show()


### 8.2 Across Specialties and Gender


In [ ]:
spec_payment = df.groupby(['spec_cat', 'gender_label']).agg(
    mean_pay=('amt', 'mean'),
    median_pay=('amt', 'median')
).reset_index()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Mean Payment (2024 dollars)', 'Median Payment (2024 dollars)']
)
for gender in ['Male', 'Female']:
    subset = spec_payment[spec_payment['gender_label'] == gender]
    fig.add_trace(
        go.Bar(x=subset['spec_cat'], y=subset['mean_pay'], name=gender, legendgroup=gender, showlegend=True),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(x=subset['spec_cat'], y=subset['median_pay'], name=gender, legendgroup=gender, showlegend=False),
        row=1, col=2
    )
fig.update_layout(title='Payment by Specialty and Gender', barmode='group', height=400)
fig.show()


### 8.3 Across Seniority and Gender


In [ ]:
seniority_payment = per_doc.groupby(['seniority', 'gender_label']).agg(
    mean_avg_pay=('avg_pay', 'mean')
).reset_index()

fig = px.bar(
    seniority_payment, x='seniority', y='mean_avg_pay',
    color='gender_label', barmode='group',
    title='Mean Payment per Physician by Seniority and Gender (2024 dollars)',
    labels={'mean_avg_pay': 'Mean Avg Payment', 'seniority': 'Seniority', 'gender_label': 'Gender'},
    category_orders={'seniority': ['1 yr', '2-3 yr', '4-5 yr', '6-8 yr', '9-11 yr']}
)
fig.show()


---
## 9. Statistical Testing

H0: No significant difference in payment distributions between genders.

H1: Female physicians receive systematically lower payments.


In [ ]:
male_totals = per_doc[per_doc['gender'] == 'M']['total']
female_totals = per_doc[per_doc['gender'] == 'F']['total']

print(f'Male physicians:   n={len(male_totals):,}, mean=${male_totals.mean():,.2f}, median=${male_totals.median():,.2f}')
print(f'Female physicians: n={len(female_totals):,}, mean=${female_totals.mean():,.2f}, median=${female_totals.median():,.2f}')

u_stat, mw_pval = stats.mannwhitneyu(female_totals, male_totals, alternative='less')
t_stat, tt_pval = stats.ttest_ind(female_totals, male_totals, equal_var=False, alternative='less')
pooled_std = np.sqrt((male_totals.std()**2 + female_totals.std()**2) / 2)
effect_size = (male_totals.mean() - female_totals.mean()) / pooled_std

print(f'\nMann-Whitney U: U={u_stat:,.0f}, p={mw_pval:.2e}')
print(f'Welch t-test:   t={t_stat:.4f}, p={tt_pval:.2e}')
print(f'Cohen d:        {effect_size:.4f}')


---
## 10. Control Models (Prove Gender Significance)

Minimal features only: gender, specialty, and year. The purpose is to isolate
whether gender has statistically significant predictive power on its own.

### 10A. Control Linear Models (predict payment, gender is a required feature)


In [ ]:
# Prepare features for control models
ctrl = df[['gender', 'amt', 'spec_cat', 'program_year']].dropna().copy()
ctrl['is_female'] = (ctrl['gender'] == 'F').astype(int)
ctrl['log_amt'] = np.log1p(ctrl['amt'])

ctrl_features = pd.get_dummies(
    ctrl[['is_female', 'program_year']],
    columns=['program_year'], prefix='yr'
)
ctrl_features = pd.concat([
    ctrl_features,
    pd.get_dummies(ctrl['spec_cat'], prefix='spec', drop_first=True)
], axis=1)

# Sample for speed
if len(ctrl_features) > 300000:
    sample_idx = ctrl_features.sample(300000, random_state=42).index
    X_ctrl_reg = ctrl_features.loc[sample_idx]
    y_ctrl_reg = ctrl['log_amt'].loc[sample_idx]
else:
    X_ctrl_reg = ctrl_features
    y_ctrl_reg = ctrl['log_amt']

Xtr, Xte, ytr, yte = train_test_split(X_ctrl_reg, y_ctrl_reg, test_size=0.2, random_state=42)
scaler = StandardScaler()
Xtr_scaled = scaler.fit_transform(Xtr)
Xte_scaled = scaler.transform(Xte)

ctrl_linear_models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.01, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000),
    'XGBoost': xgb.XGBRegressor(
        n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42, verbosity=0
    )
}

ctrl_reg_results = []
ctrl_gender_coefficients = {}

for name, model in ctrl_linear_models.items():
    needs_scaling = name in ['Linear Regression', 'Ridge', 'Lasso', 'ElasticNet']
    xt = Xtr_scaled if needs_scaling else Xtr
    xv = Xte_scaled if needs_scaling else Xte

    model.fit(xt, ytr)
    predictions = model.predict(xv)
    r2 = r2_score(yte, predictions)
    rmse = np.sqrt(mean_squared_error(yte, predictions))

    ctrl_reg_results.append({'Model': name, 'R2': round(r2, 4), 'RMSE': round(rmse, 4)})

    if hasattr(model, 'coef_'):
        female_idx = list(X_ctrl_reg.columns).index('is_female')
        ctrl_gender_coefficients[name] = model.coef_[female_idx]

    print(f'  {name}: R2={r2:.4f}, RMSE={rmse:.4f}')

ctrl_reg_df = pd.DataFrame(ctrl_reg_results).sort_values('R2', ascending=False)

print('\nGender coefficients (control models):')
for name, coef in ctrl_gender_coefficients.items():
    pct_change = 100 * (np.exp(coef) - 1)
    print(f'  {name}: {coef:.4f} ({pct_change:.1f}%)')


### 10B. Control Logistic Models (predict gender, payment is a required feature)


In [ ]:
X_ctrl_clf = ctrl_features.drop(columns=['is_female']).copy()
X_ctrl_clf['amt'] = ctrl['amt'].loc[X_ctrl_clf.index]
y_ctrl_clf = ctrl['is_female'].loc[X_ctrl_clf.index]

Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(
    X_ctrl_clf, y_ctrl_clf, test_size=0.2, random_state=42, stratify=y_ctrl_clf
)
scaler_c = StandardScaler()
Xtr_c_scaled = scaler_c.fit_transform(Xtr_c)
Xte_c_scaled = scaler_c.transform(Xte_c)

ctrl_logistic_models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Ridge Logistic': LogisticRegression(penalty='l2', C=0.1, max_iter=500, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss', verbosity=0
    )
}

ctrl_clf_results = []
ctrl_roc_data = {}

for name, model in ctrl_logistic_models.items():
    needs_scaling = 'Logistic' in name or 'Ridge' in name
    xt = Xtr_c_scaled if needs_scaling else Xtr_c
    xv = Xte_c_scaled if needs_scaling else Xte_c

    model.fit(xt, ytr_c)
    preds = model.predict(xv)
    probs = model.predict_proba(xv)[:, 1]
    auc = roc_auc_score(yte_c, probs)
    fpr, tpr, _ = roc_curve(yte_c, probs)
    ctrl_roc_data[name] = (fpr, tpr, auc)
    ctrl_clf_results.append({
        'Model': name, 'AUC': round(auc, 4),
        'Accuracy': round(accuracy_score(yte_c, preds), 4),
        'F1': round(f1_score(yte_c, preds), 4)
    })
    print(f'  {name}: AUC={auc:.4f}')

ctrl_clf_df = pd.DataFrame(ctrl_clf_results).sort_values('AUC', ascending=False)


---
## 11. All-Variable Models (Best Prediction)

Full feature set: gender, specialty, state, company tier, payment type, year.

### 11A. All-Variable Linear Models (predict payment, gender is required)


In [ ]:
full = df[['gender', 'amt', 'spec_cat', 'program_year', 'state', 'company', 'pay_type']].dropna().copy()
full['is_female'] = (full['gender'] == 'F').astype(int)
full['log_amt'] = np.log1p(full['amt'])
full['company_tier'] = full['company'].apply(
    lambda c: 'top5' if c in top_5_companies else 'top20' if c in top_20_companies else 'other'
)
full['state_group'] = full['state'].apply(lambda s: s if s in top_15_states else 'Other')
full['pay_type_group'] = full['pay_type'].apply(lambda p: p if p in top_10_pay_types else 'Other')

full_features = pd.DataFrame({'is_female': full['is_female']})
full_features = pd.concat([
    full_features,
    pd.get_dummies(full['program_year'].astype(str), prefix='yr', drop_first=True)
], axis=1)
for col in ['spec_cat', 'company_tier', 'state_group', 'pay_type_group']:
    full_features = pd.concat([
        full_features,
        pd.get_dummies(full[col], prefix=col[:4], drop_first=True)
    ], axis=1)

if len(full_features) > 300000:
    sample_idx2 = full_features.sample(300000, random_state=42).index
    full_features = full_features.loc[sample_idx2]
    full = full.loc[sample_idx2]

# Regression
X_full_reg = full_features.copy()
y_full_reg = full['log_amt']
Xtr2, Xte2, ytr2, yte2 = train_test_split(X_full_reg, y_full_reg, test_size=0.2, random_state=42)
scaler2 = StandardScaler()
Xtr2_s = scaler2.fit_transform(Xtr2)
Xte2_s = scaler2.transform(Xte2)

print(f'All-variable features: {X_full_reg.shape[1]} columns')

allvar_linear_models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.01, max_iter=5000),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBRegressor(
        n_estimators=150, max_depth=6, learning_rate=0.1, random_state=42, verbosity=0
    )
}

allvar_reg_results = []
allvar_gender_coefficients = {}

for name, model in allvar_linear_models.items():
    needs_scaling = name in ['Linear Regression', 'Ridge', 'Lasso']
    xt = Xtr2_s if needs_scaling else Xtr2
    xv = Xte2_s if needs_scaling else Xte2

    model.fit(xt, ytr2)
    predictions = model.predict(xv)
    r2 = r2_score(yte2, predictions)
    rmse = np.sqrt(mean_squared_error(yte2, predictions))
    allvar_reg_results.append({'Model': name, 'R2': round(r2, 4), 'RMSE': round(rmse, 4)})

    if hasattr(model, 'coef_'):
        allvar_gender_coefficients[name] = model.coef_[list(X_full_reg.columns).index('is_female')]

    print(f'  {name}: R2={r2:.4f}')

allvar_reg_df = pd.DataFrame(allvar_reg_results).sort_values('R2', ascending=False)

print('\nGender coefficients (all-variable models):')
for name, coef in allvar_gender_coefficients.items():
    print(f'  {name}: {coef:.4f} ({100 * (np.exp(coef) - 1):.1f}%)')


### 11B. All-Variable Logistic Models (predict gender, payment is required)


In [ ]:
X_full_clf = full_features.drop(columns=['is_female']).copy()
X_full_clf['amt'] = full['amt'].values
y_full_clf = full_features['is_female']

Xtr2c, Xte2c, ytr2c, yte2c = train_test_split(
    X_full_clf, y_full_clf, test_size=0.2, random_state=42, stratify=y_full_clf
)
scaler2c = StandardScaler()
Xtr2c_s = scaler2c.fit_transform(Xtr2c)
Xte2c_s = scaler2c.transform(Xte2c)

allvar_logistic_models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Ridge Logistic': LogisticRegression(penalty='l2', C=0.1, max_iter=500, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=150, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss', verbosity=0
    )
}

allvar_clf_results = []
allvar_roc_data = {}

for name, model in allvar_logistic_models.items():
    needs_scaling = 'Logistic' in name or 'Ridge' in name
    xt = Xtr2c_s if needs_scaling else Xtr2c
    xv = Xte2c_s if needs_scaling else Xte2c

    model.fit(xt, ytr2c)
    preds = model.predict(xv)
    probs = model.predict_proba(xv)[:, 1]
    auc = roc_auc_score(yte2c, probs)
    fpr, tpr, _ = roc_curve(yte2c, probs)
    allvar_roc_data[name] = (fpr, tpr, auc)
    allvar_clf_results.append({
        'Model': name, 'AUC': round(auc, 4),
        'Accuracy': round(accuracy_score(yte2c, preds), 4),
        'F1': round(f1_score(yte2c, preds), 4)
    })
    print(f'  {name}: AUC={auc:.4f}')

allvar_clf_df = pd.DataFrame(allvar_clf_results).sort_values('AUC', ascending=False)


---
## 12. Model Evaluation Dashboard


### 12.1 Control Model Comparison


In [ ]:
fig = px.bar(
    ctrl_reg_df, x='Model', y='R2',
    title='Control Models: Regression R-squared (Minimal Features)'
)
fig.show()

fig = go.Figure()
for name, (fpr, tpr, auc) in ctrl_roc_data.items():
    fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f'{name} ({auc:.3f})', mode='lines'))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], name='Chance', line=dict(dash='dash', color='gray')))
fig.update_layout(
    title='Control Models: Classification ROC Curves',
    xaxis_title='False Positive Rate', yaxis_title='True Positive Rate',
    height=450
)
fig.show()


### 12.2 All-Variable Model Comparison


In [ ]:
fig = px.bar(
    allvar_reg_df, x='Model', y='R2',
    title='All-Variable Models: Regression R-squared'
)
fig.show()

fig = go.Figure()
for name, (fpr, tpr, auc) in allvar_roc_data.items():
    fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f'{name} ({auc:.3f})', mode='lines'))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], name='Chance', line=dict(dash='dash', color='gray')))
fig.update_layout(
    title='All-Variable Models: Classification ROC Curves',
    xaxis_title='False Positive Rate', yaxis_title='True Positive Rate',
    height=450
)
fig.show()


### 12.3 Side-by-Side: Control vs All-Variable


In [ ]:
# Regression comparison
comparison_reg = ctrl_reg_df[['Model', 'R2']].rename(columns={'R2': 'Control'}).merge(
    allvar_reg_df[['Model', 'R2']].rename(columns={'R2': 'All Variables'}),
    on='Model', how='outer'
)
comparison_reg['Improvement'] = comparison_reg['All Variables'] - comparison_reg['Control']
print('Regression: Control vs All-Variable')
print(comparison_reg.to_string(index=False))

melted_reg = comparison_reg.melt(
    id_vars='Model', value_vars=['Control', 'All Variables'],
    var_name='Model Set', value_name='R2'
)
fig = px.bar(
    melted_reg, x='Model', y='R2', color='Model Set', barmode='group',
    title='Regression R-squared: Control vs All-Variable'
)
fig.show()

# Classification comparison
comparison_clf = ctrl_clf_df[['Model', 'AUC']].rename(columns={'AUC': 'Control'}).merge(
    allvar_clf_df[['Model', 'AUC']].rename(columns={'AUC': 'All Variables'}),
    on='Model', how='outer'
)
comparison_clf['Improvement'] = comparison_clf['All Variables'] - comparison_clf['Control']
print('\nClassification: Control vs All-Variable')
print(comparison_clf.to_string(index=False))

melted_clf = comparison_clf.melt(
    id_vars='Model', value_vars=['Control', 'All Variables'],
    var_name='Model Set', value_name='AUC'
)
fig = px.bar(
    melted_clf, x='Model', y='AUC', color='Model Set', barmode='group',
    title='Classification AUC: Control vs All-Variable'
)
fig.add_hline(y=0.5, line_dash='dash', annotation_text='Chance')
fig.show()


### 12.4 Gender Coefficient Gauge: Control vs All-Variable

If the coefficient stays negative and significant after adding all controls,
gender has independent explanatory power beyond compositional factors.


In [ ]:
ctrl_avg_coef = np.mean(list(ctrl_gender_coefficients.values())) if ctrl_gender_coefficients else 0
allvar_avg_coef = np.mean(list(allvar_gender_coefficients.values())) if allvar_gender_coefficients else 0

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'indicator'}, {'type': 'indicator'}]],
    subplot_titles=['Control Models', 'All-Variable Models']
)

for i, (label, value) in enumerate([
    ('Control', ctrl_avg_coef),
    ('All Variables', allvar_avg_coef)
]):
    fig.add_trace(
        go.Indicator(
            mode='gauge+number',
            value=value,
            title={'text': 'is_female coefficient'},
            number={'valueformat': '.4f'},
            gauge={
                'axis': {'range': [-0.5, 0.1]},
                'bar': {'color': 'crimson'},
                'threshold': {
                    'line': {'color': 'black', 'width': 3},
                    'thickness': 0.75,
                    'value': 0
                }
            }
        ),
        row=1, col=i + 1
    )

fig.update_layout(height=350, title='is_female Coefficient: Control vs All-Variable')
fig.show()

print(f'Control avg coefficient:      {ctrl_avg_coef:.4f} ({100 * (np.exp(ctrl_avg_coef) - 1):.1f}%)')
print(f'All-variable avg coefficient: {allvar_avg_coef:.4f} ({100 * (np.exp(allvar_avg_coef) - 1):.1f}%)')
delta = allvar_avg_coef - ctrl_avg_coef
direction = 'shrinks' if abs(allvar_avg_coef) < abs(ctrl_avg_coef) else 'persists'
print(f'The gender effect {direction} after adding controls (delta: {delta:.4f})')


---
## 13. Streamlit Dashboard

This cell writes a standalone app.py for deployment on Streamlit Community Cloud.

**Deployment steps:**
1. Download `biotech_payments_2014_2024.csv` and `app.py` from Colab
2. Push both files to a GitHub repository
3. Go to streamlit.io/cloud and connect the repo
4. Set the main file to `app.py`
5. The dashboard will be live in about two minutes


In [ ]:
app_code = '''import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Biotech Gender Pay Parity", layout="wide")

@st.cache_data
def load():
    df = pd.read_csv("biotech_payments_2014_2024.csv")
    df["gender_label"] = df["gender"].map({"M": "Male", "F": "Female"})
    SPECS = {"Surgery": ["surgery", "surgical"], "Oncology": ["oncology"],
             "Cardiology": ["cardiology"], "Neurology": ["neurology"]}
    def classify(s):
        if not isinstance(s, str): return None
        lower = s.lower()
        for cat, kws in SPECS.items():
            if any(k in lower for k in kws): return cat
        return None
    df["spec_cat"] = df["specialty_raw"].apply(classify)
    return df[df.spec_cat.notna()]

df = load()
m = df[df.gender == "M"]["amt"]
f = df[df.gender == "F"]["amt"]

c1, c2, c3, c4 = st.columns(4)
f_pct = round(100 * len(df[df.gender == "F"]) / len(df), 1)
f_amt_pct = round(100 * f.sum() / df.amt.sum(), 1)
c1.metric("Records", f"{len(df):,}", f"{f_pct}% F")
c2.metric("Total $", f"${df.amt.sum() / 1e6:.0f}M", f"{f_amt_pct}% F")
c3.metric("F/M Median Overall", f"{f.median() / m.median():.3f}")
c4.metric("F/M Median Biotech", f"{f.median() / m.median():.3f}")
st.divider()

tab1, tab2, tab3 = st.tabs(["Gender Distribution", "Payment Distribution", "Model Results"])

with tab1:
    sg = df.groupby(["state", "gender_label"]).npi.nunique().reset_index(name="docs")
    sp = sg.pivot_table(index="state", columns="gender_label", values="docs", fill_value=0).reset_index()
    if "Female" in sp.columns and "Male" in sp.columns:
        sp["pct_f"] = (100 * sp["Female"] / (sp.Female + sp.Male)).round(1)
        sp = sp[sp.state.str.len() == 2]
        fig = px.choropleth(sp, locations="state", locationmode="USA-states",
            color="pct_f", scope="usa", color_continuous_scale="RdBu", range_color=[25, 50])
        st.plotly_chart(fig, use_container_width=True)

with tab2:
    samp = df.sample(min(100000, len(df)), random_state=42)
    fig = px.histogram(samp, x="amt", color="gender_label", nbins=60, log_y=True,
        barmode="overlay", opacity=0.6)
    st.plotly_chart(fig, use_container_width=True)

with tab3:
    st.info("Model evaluation charts are in the Colab notebook.")
'''

with open('/content/app.py', 'w') as f:
    f.write(app_code)
print('app.py written to /content/app.py')


---
## Summary

| Component | Detail |
| --- | --- |
| Data | CMS Open Payments 2014 to 2024, inflation-adjusted, NPPES gender only |
| CSV | biotech_payments_2014_2024.csv (raw files discarded after processing) |
| KPIs | Records (%F), Total $ (%F), F/M Median Overall, F/M Median Biotech |
| Gender charts | Choropleth map, specialty bars, seniority bars |
| Payment charts | R/Y/G choropleth, specialty+gender, seniority+gender |
| Control models | 5 linear + 5 logistic (gender + specialty + year only) |
| All-variable models | 5 linear + 5 logistic (adds state, company, payment type) |
| Evaluation | Control comparison, all-var comparison, side-by-side, gauge chart |
| Dashboard | Streamlit app.py for Community Cloud deployment |


In [ ]:
print('Analysis complete.')
con.close()
